In [2]:
# Exécute cette cellule pour installer RDKit dans ton environnement VS Code
import sys
!{sys.executable} -m pip install rdkit

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Wissal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import os
import pickle
import requests
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors, rdMolAlign

print("✅ Environnement configuré et prêt.")

✅ Environnement configuré et prêt.


In [4]:
def generate_3d_structure(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    mol_with_hs = Chem.AddHs(mol) # Ajout des Hydrogènes
    params = AllChem.ETKDGv3()
    status = AllChem.EmbedMolecule(mol_with_hs, params)
    return mol_with_hs if status == 0 else None

def optimize_3d_structure(mol):
    if mol is None: return None
    mol_opt = Chem.Mol(mol)
    # Minimisation de l'énergie avec le champ de force MMFF94
    AllChem.MMFFOptimizeMolecule(mol_opt)
    return mol_opt

def get_smiles_from_name(name):
    """Traduit un nom de médicament en SMILES via l'API NCI."""
    try:
        url = f"https://cactus.nci.nih.gov/chemical/structure/{name}/smiles"
        response = requests.get(url, timeout=10)
        return response.text.strip() if response.status_code == 200 else None
    except:
        return None

In [5]:
def calculate_metrics(mol_brute, mol_opt):
    rmsd = rdMolAlign.GetBestRMS(mol_opt, mol_brute)
    problems = Chem.DetectChemistryProblems(mol_opt)
    
    mp = AllChem.MMFFGetMoleculeProperties(mol_opt)
    ff = AllChem.MMFFGetMoleculeForceField(mol_opt, mp)
    energy = ff.CalcEnergy()
    
    metrics = {
        "rmsd_angstrom": round(rmsd, 4),
        "chemically_valid": len(problems) == 0,
        "energy_total": round(energy, 2),
        "is_converged": ff.Minimize() == 0
    }
    return metrics

def save_final_package(mol, name, smiles, metrics):
    filename = f"agent_3d_{name.lower().replace(' ', '_')}.pkl"
    package = {
        "identity": {"name": name, "smiles": smiles},
        "quality": metrics,
        "rdkit_obj": mol
    }
    with open(filename, "wb") as f:
        pickle.dump(package, f)
    return filename

In [6]:
import py3Dmol

def show_molecule(mol, label):
    mb = Chem.MolToMolBlock(mol)
    view = py3Dmol.view(width=400, height=400)
    view.addModel(mb, 'sdf')
    view.setStyle({'stick': {}, 'sphere': {'radius': 0.3}})
    view.zoomTo()
    return view.show()

In [7]:
def start_agent():
    print("\n--- AGENT 3D PRINTER v1.0 ---")
    user_input = input("Entrez le NOM (ex: Aspirin) ou le SMILES : ").strip()
    
    # 1. Identification
    if any(c in user_input for c in "=()#"):
        smiles = user_input
        name = "Molecule_Perso"
    else:
        print(f"🔍 Recherche du SMILES pour '{user_input}'...")
        smiles = get_smiles_from_name(user_input)
        name = user_input

    if not smiles:
        print("❌ Erreur : Structure introuvable.")
        return

    # 2. Pipeline
    print("🚀 Génération 3D et Optimisation...")
    mol_brute = generate_3d_structure(smiles)
    mol_final = optimize_3d_structure(mol_brute)
    
    # 3. Validation et Export
    stats = calculate_metrics(mol_brute, mol_final)
    fname = save_final_package(mol_final, name, smiles, stats)
    
    print(f"✅ Terminé ! Fichier : {fname}")
    print(f"📊 Qualité (RMSD) : {stats['rmsd_angstrom']} Å")
    
    # 4. Vue 3D
    return show_molecule(mol_final, name)

# LANCEMENT
start_agent()


--- AGENT 3D PRINTER v1.0 ---
🔍 Recherche du SMILES pour 'Paracetamol'...
🚀 Génération 3D et Optimisation...
✅ Terminé ! Fichier : agent_3d_paracetamol.pkl
📊 Qualité (RMSD) : 0.7757 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.